In [2]:
from itertools import combinations
from collections import Counter


RANKS = "23456789TJQKA"
SUITS = "cdhs"

deck = [r + s for r in RANKS for s in SUITS]


def evaluate_7(cards):
    """
    Evaluate a 7-card Texas Hold'em hand.

    Returns a tuple where a larger tuple means a stronger hand.
    """

    ranks = [RANKS.index(c[0]) + 2 for c in cards]
    suits = [c[1] for c in cards]

    counts = Counter(ranks)

    # -------------------------
    # Find flush
    # -------------------------
    suit_counts = Counter(suits)

    flush_suit = next(
        (s for s, count in suit_counts.items() if count >= 5),
        None
    )

    # -------------------------
    # Find straight
    # -------------------------
    unique_ranks = set(ranks)

    # Ace can also be used as a 1
    if 14 in unique_ranks:
        unique_ranks.add(1)

    straight_high = None

    for high in range(14, 4, -1):
        if all(r in unique_ranks for r in range(high - 4, high + 1)):
            straight_high = high
            break

    # -------------------------
    # Straight flush
    # -------------------------
    if flush_suit:

        flush_ranks = [
            RANKS.index(c[0]) + 2
            for c in cards
            if c[1] == flush_suit
        ]

        flush_ranks = set(flush_ranks)

        if 14 in flush_ranks:
            flush_ranks.add(1)

        for high in range(14, 4, -1):
            if all(r in flush_ranks for r in range(high - 4, high + 1)):
                return (8, high)

    # -------------------------
    # Four of a kind
    # -------------------------
    fours = sorted(
        [r for r, n in counts.items() if n == 4],
        reverse=True
    )

    if fours:
        four = fours[0]
        kicker = max(r for r in ranks if r != four)

        return (7, four, kicker)

    # -------------------------
    # Full house
    # -------------------------
    trips = sorted(
        [r for r, n in counts.items() if n >= 3],
        reverse=True
    )

    pairs = sorted(
        [r for r, n in counts.items() if n >= 2],
        reverse=True
    )

    if trips:
        trip = trips[0]

        # Need another pair/trips for the full house
        remaining_pairs = [
            r for r in pairs
            if r != trip
        ]

        if remaining_pairs:
            return (6, trip, remaining_pairs[0])

    # -------------------------
    # Flush
    # -------------------------
    if flush_suit:

        flush_cards = sorted(
            [
                RANKS.index(c[0]) + 2
                for c in cards
                if c[1] == flush_suit
            ],
            reverse=True
        )

        return (5, *flush_cards[:5])

    # -------------------------
    # Straight
    # -------------------------
    if straight_high:
        return (4, straight_high)

    # -------------------------
    # Three of a kind
    # -------------------------
    if trips:
        trip = trips[0]

        kickers = sorted(
            [r for r in ranks if r != trip],
            reverse=True
        )

        return (3, trip, *kickers[:2])

    # -------------------------
    # Two pair
    # -------------------------
    if len(pairs) >= 2:

        p1, p2 = pairs[:2]

        kicker = max(
            r for r in ranks
            if r != p1 and r != p2
        )

        return (2, p1, p2, kicker)

    # -------------------------
    # One pair
    # -------------------------
    if len(pairs) == 1:

        pair = pairs[0]

        kickers = sorted(
            [r for r in ranks if r != pair],
            reverse=True
        )

        return (1, pair, *kickers[:3])

    # -------------------------
    # High card
    # -------------------------
    return (0, *sorted(ranks, reverse=True)[:5])

# Function
def poker_odds(hand1, hand2):

    used = set(hand1 + hand2)

    remaining = [
        card for card in deck
        if card not in used
    ]

    wins = 0
    ties = 0
    losses = 0

    for board in combinations(remaining, 5):

        score1 = evaluate_7(hand1 + list(board))
        score2 = evaluate_7(hand2 + list(board))

        if score1 > score2:
            wins += 1

        elif score1 == score2:
            ties += 1

        else:
            losses += 1

    total = wins + ties + losses

    return {
        "win": wins / total,
        "tie": ties / total,
        "lose": losses / total,
        "equity": (wins + ties / 2) / total
    }

In [3]:
# Test
odds = poker_odds(
    ["As", "Ah"],
    ["Ks", "Kh"]
)

print(odds)

{'win': 0.8236481372466571, 'tie': 0.005435950625589848, 'lose': 0.170915912127753, 'equity': 0.826366112559452}
